# Evaluating Model Outputs

We can evaluate a model's confidence in its results by using perplexity. Perplexity is a measure of uncertainty that can be calculated by exponentiating the negative of the average of the logprobs. 

+ Perplexity can be used to assess the result of an individual model run.
+ It can also be used to compare the relative confidence of results between model runs. 

Low perplexity or high confidence does not guarantee accuracy, but it can be a helpful signal when paired with other evaluation metrics. 

In [1]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets
import sys
sys.path.append('../../05_src/')

In [2]:
from utils.clients import get_client
from IPython.display import display, Markdown
import numpy as np

client = get_client()

In [3]:
prompts = [
    # Low perplexity: Clear topic, common structure, highly predictable vocabulary
    "Explain how photosynthesis works in simple terms.",
    # Medium preplexity: Narrative freedom, but familiar theme and constraints.
    "Write a short story about a traveler who realizes the journey mattered more than the destination.",
    # High perplexity: Abstract concept, creative freedom, unpredictable vocabulary
    "Describe the taste of a color that only exists for one second at dusk, using metaphors from mathematics and weather."
]

In [4]:
def get_completion(
    input: list[dict[str, str]],
    model: str = "gpt-4o-mini",
    max_tokens=500,
    temperature=0,
    tools=None,
    logprobs=None,  # whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message..
    top_logprobs=None,
) -> str:
    params = {
        "model": model,
        "input": input,
        "max_output_tokens": max_tokens,
        "temperature": temperature,
        "tools": tools,
        "include": ["message.output_text.logprobs"] if logprobs else [],
        "top_logprobs": top_logprobs,
    }
    if tools:
        params["tools"] = tools

    completion = client.responses.create(**params)
    return completion

In [5]:
for k, prompt in enumerate(prompts):
    API_RESPONSE = get_completion(
        [{"role": "user", "content": prompt}],
        model="gpt-4o-mini",
        logprobs=True,
    )
    token_data = API_RESPONSE.output[0].content[0].logprobs
    response_text = API_RESPONSE.output[0].content[0].text
    lp_values = [t.logprob for t in token_data]
    perplexity_score = np.exp(-np.mean(lp_values))

    rows = [
        f"| `{t.token.replace('|', chr(9474)).replace(chr(10), '↵').replace(chr(96), chr(39))}` "
        f"| {t.logprob:.4f} | {np.exp(t.logprob)*100:.1f}% |"
        for t in token_data
    ]
    table = "\n".join([
        "| Token | Logprob | Linear Prob |",
        "|:------|--------:|------------:|",
    ] + rows)

    display(Markdown(f"### Prompt {k+1}\n_{prompt}_"))
    display(Markdown(f"**Response:**\n\n{response_text}"))
    display(Markdown(table))
    display(Markdown(f"**Perplexity:** `{perplexity_score:.2f}`\n\n---"))

### Prompt 1
_Explain how photosynthesis works in simple terms._

**Response:**

Photosynthesis is the process that plants, algae, and some bacteria use to make their own food. Here’s how it works in simple terms:

1. **Sunlight**: Plants take in sunlight using a green pigment called chlorophyll, which is found in their leaves.

2. **Water**: Plants absorb water from the soil through their roots.

3. **Carbon Dioxide**: Plants take in carbon dioxide from the air through tiny openings in their leaves called stomata.

4. **Making Food**: Using the energy from sunlight, plants combine water and carbon dioxide to create glucose (a type of sugar) and oxygen. The glucose is used as food for energy and growth.

5. **Oxygen Release**: The oxygen produced during this process is released into the air, which is essential for humans and animals to breathe.

In summary, photosynthesis is how plants turn sunlight, water, and carbon dioxide into food and oxygen!

| Token | Logprob | Linear Prob |
|:------|--------:|------------:|
| `Photos` | -0.0793 | 92.4% |
| `ynthesis` | 0.0000 | 100.0% |
| ` is` | -0.0000 | 100.0% |
| ` the` | -0.0485 | 95.3% |
| ` process` | -0.0041 | 99.6% |
| ` that` | -0.2279 | 79.6% |
| ` plants` | -0.0039 | 99.6% |
| `,` | -0.5766 | 56.2% |
| ` algae` | -0.0262 | 97.4% |
| `,` | 0.0000 | 100.0% |
| ` and` | 0.0000 | 100.0% |
| ` some` | -0.0000 | 100.0% |
| ` bacteria` | -0.0001 | 100.0% |
| ` use` | -0.0000 | 100.0% |
| ` to` | 0.0000 | 100.0% |
| ` make` | -0.1376 | 87.1% |
| ` their` | -0.0067 | 99.3% |
| ` own` | -0.0789 | 92.4% |
| ` food` | -0.0000 | 100.0% |
| `.` | -0.5787 | 56.1% |
| ` Here` | -0.2049 | 81.5% |
| `’s` | -0.0001 | 100.0% |
| ` how` | -0.1002 | 90.5% |
| ` it` | 0.0000 | 100.0% |
| ` works` | -0.0000 | 100.0% |
| ` in` | -0.0238 | 97.6% |
| ` simple` | -0.0001 | 100.0% |
| ` terms` | -0.0015 | 99.8% |
| `:↵↵` | -0.0000 | 100.0% |
| `1` | -0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | -0.0000 | 100.0% |
| `Sun` | -0.7172 | 48.8% |
| `light` | -0.0003 | 100.0% |
| `**` | -0.1083 | 89.7% |
| `:` | -0.0000 | 100.0% |
| ` Plants` | -0.0010 | 99.9% |
| ` take` | -0.7735 | 46.1% |
| ` in` | -0.0006 | 99.9% |
| ` sunlight` | -0.0190 | 98.1% |
| ` using` | -0.5048 | 60.4% |
| ` a` | -0.2050 | 81.5% |
| ` green` | -0.0332 | 96.7% |
| ` pigment` | -0.0021 | 99.8% |
| ` called` | -0.0381 | 96.3% |
| ` chlor` | -0.0000 | 100.0% |
| `ophyll` | 0.0000 | 100.0% |
| `,` | -0.5955 | 55.1% |
| ` which` | -0.6916 | 50.1% |
| ` is` | -0.0002 | 100.0% |
| ` found` | -0.4741 | 62.2% |
| ` in` | -0.0136 | 98.7% |
| ` their` | -0.0023 | 99.8% |
| ` leaves` | -0.0000 | 100.0% |
| `.↵↵` | -0.0032 | 99.7% |
| `2` | 0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Water` | -0.4073 | 66.5% |
| `**` | -0.3156 | 72.9% |
| `:` | 0.0000 | 100.0% |
| ` Plants` | -0.6429 | 52.6% |
| ` absorb` | -0.0062 | 99.4% |
| ` water` | -0.0000 | 100.0% |
| ` from` | -0.0341 | 96.7% |
| ` the` | -0.0001 | 100.0% |
| ` soil` | -0.0142 | 98.6% |
| ` through` | -0.0000 | 100.0% |
| ` their` | -0.0000 | 100.0% |
| ` roots` | -0.0000 | 100.0% |
| `.↵↵` | -0.0010 | 99.9% |
| `3` | 0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Carbon` | -0.0004 | 100.0% |
| ` D` | -0.0052 | 99.5% |
| `ioxide` | -0.0000 | 100.0% |
| `**` | -0.0002 | 100.0% |
| `:` | 0.0000 | 100.0% |
| ` Plants` | -0.0429 | 95.8% |
| ` take` | -0.0411 | 96.0% |
| ` in` | -0.0004 | 100.0% |
| ` carbon` | -0.0041 | 99.6% |
| ` dioxide` | 0.0000 | 100.0% |
| ` from` | -0.0556 | 94.6% |
| ` the` | -0.0000 | 100.0% |
| ` air` | -0.0000 | 100.0% |
| ` through` | -0.0002 | 100.0% |
| ` tiny` | -0.1004 | 90.5% |
| ` openings` | -0.0113 | 98.9% |
| ` in` | -0.0148 | 98.5% |
| ` their` | -0.0003 | 100.0% |
| ` leaves` | 0.0000 | 100.0% |
| ` called` | -0.0045 | 99.5% |
| ` stom` | -0.0001 | 100.0% |
| `ata` | -0.0000 | 100.0% |
| `.↵↵` | -0.0000 | 100.0% |
| `4` | -0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Making` | -1.1345 | 32.2% |
| ` Food` | -0.0044 | 99.6% |
| `**` | -0.0001 | 100.0% |
| `:` | -0.0001 | 100.0% |
| ` Using` | -0.0531 | 94.8% |
| ` the` | -0.4750 | 62.2% |
| ` energy` | -0.2023 | 81.7% |
| ` from` | -0.0000 | 100.0% |
| ` sunlight` | -0.0041 | 99.6% |
| `,` | -0.0000 | 100.0% |
| ` plants` | -0.0081 | 99.2% |
| ` combine` | -0.1403 | 86.9% |
| ` water` | -0.1688 | 84.5% |
| ` and` | -0.0001 | 100.0% |
| ` carbon` | 0.0000 | 100.0% |
| ` dioxide` | 0.0000 | 100.0% |
| ` to` | -0.0030 | 99.7% |
| ` create` | -0.3005 | 74.0% |
| ` glucose` | -0.0246 | 97.6% |
| ` (` | -0.0382 | 96.3% |
| `a` | -0.0004 | 100.0% |
| ` type` | -0.0101 | 99.0% |
| ` of` | 0.0000 | 100.0% |
| ` sugar` | -0.0000 | 100.0% |
| `)` | -0.0998 | 90.5% |
| ` and` | -0.0020 | 99.8% |
| ` oxygen` | -0.0008 | 99.9% |
| `.` | -0.2017 | 81.7% |
| ` The` | -0.8216 | 44.0% |
| ` glucose` | -0.5041 | 60.4% |
| ` is` | -0.5940 | 55.2% |
| ` used` | -0.3827 | 68.2% |
| ` as` | -0.2396 | 78.7% |
| ` food` | -0.0234 | 97.7% |
| ` for` | -0.0537 | 94.8% |
| ` energy` | -0.6942 | 49.9% |
| ` and` | -0.0187 | 98.1% |
| ` growth` | -0.0000 | 100.0% |
| `.↵↵` | -0.3490 | 70.5% |
| `5` | -0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `O` | -0.1537 | 85.8% |
| `xygen` | -0.0000 | 100.0% |
| ` Release` | -0.0408 | 96.0% |
| `**` | -0.0000 | 100.0% |
| `:` | 0.0000 | 100.0% |
| ` The` | -0.7222 | 48.6% |
| ` oxygen` | -0.1104 | 89.5% |
| ` produced` | -0.0228 | 97.7% |
| ` during` | -0.1293 | 87.9% |
| ` this` | -0.0325 | 96.8% |
| ` process` | -0.0000 | 100.0% |
| ` is` | -0.0002 | 100.0% |
| ` released` | -0.0021 | 99.8% |
| ` into` | -0.6932 | 50.0% |
| ` the` | 0.0000 | 100.0% |
| ` air` | -0.0052 | 99.5% |
| `,` | -0.0128 | 98.7% |
| ` which` | -0.0016 | 99.8% |
| ` is` | -0.0329 | 96.8% |
| ` essential` | -1.1400 | 32.0% |
| ` for` | -0.0000 | 100.0% |
| ` humans` | -0.7992 | 45.0% |
| ` and` | -0.0000 | 100.0% |
| ` animals` | -0.0284 | 97.2% |
| ` to` | -0.0010 | 99.9% |
| ` breathe` | -0.0000 | 100.0% |
| `.↵↵` | -0.0002 | 100.0% |
| `In` | -0.3145 | 73.0% |
| ` summary` | -0.1641 | 84.9% |
| `,` | -0.0047 | 99.5% |
| ` photos` | -0.0843 | 91.9% |
| `ynthesis` | 0.0000 | 100.0% |
| ` is` | -1.0465 | 35.1% |
| ` how` | -0.0611 | 94.1% |
| ` plants` | -0.0001 | 100.0% |
| ` turn` | -0.3582 | 69.9% |
| ` sunlight` | -0.0068 | 99.3% |
| `,` | -0.0233 | 97.7% |
| ` water` | -0.0001 | 100.0% |
| `,` | -0.0000 | 100.0% |
| ` and` | 0.0000 | 100.0% |
| ` carbon` | -0.0142 | 98.6% |
| ` dioxide` | 0.0000 | 100.0% |
| ` into` | 0.0000 | 100.0% |
| ` food` | -0.0052 | 99.5% |
| ` and` | -0.1527 | 85.8% |
| ` oxygen` | -0.0013 | 99.9% |
| `!` | -0.2209 | 80.2% |

**Perplexity:** `1.12`

---

### Prompt 2
_Write a short story about a traveler who realizes the journey mattered more than the destination._

...

**Perplexity:** `1.46`

---

### Prompt 3
_Describe the taste of a color that only exists for one second at dusk, using metaphors from mathematics and weather._

**Response:**

Imagine a color that tastes like the fleeting moment when twilight kisses the horizon—a blend of soft lavender and deep indigo, like the gentle curve of a parabolic arc just before it meets the ground. It’s the taste of a cool breeze, whispering secrets of a storm that never arrives, a delicate balance of potential energy suspended in the air.

This color tastes like the sweet tang of dew on grass, a fleeting probability that dances between certainty and chaos, like the unpredictable patterns of a fractal. It’s the ephemeral flavor of a gentle rain, each drop a note in a symphony of fleeting moments, harmonizing in a perfect equation of warmth and coolness.

As it fades, it leaves a trace of nostalgia, like the last digit of pi—infinitely elusive, yet profoundly present for that single second, a reminder of the beauty in transience, where every taste is a variable in the equation of dusk.

| Token | Logprob | Linear Prob |
|:------|--------:|------------:|
| `Imagine` | -0.3913 | 67.6% |
| ` a` | -0.4785 | 62.0% |
| ` color` | -0.4456 | 64.0% |
| ` that` | -0.0099 | 99.0% |
| ` tastes` | -1.4806 | 22.7% |
| ` like` | -0.0003 | 100.0% |
| ` the` | -0.2203 | 80.2% |
| ` fleeting` | -0.1911 | 82.6% |
| ` moment` | -0.2962 | 74.4% |
| ` when` | -0.1898 | 82.7% |
| ` twilight` | -0.6918 | 50.1% |
| ` kisses` | -1.8057 | 16.4% |
| ` the` | -0.0336 | 96.7% |
| ` horizon` | -0.0901 | 91.4% |
| `—a` | -0.7871 | 45.5% |
| ` blend` | -1.6397 | 19.4% |
| ` of` | -0.0231 | 97.7% |
| ` soft` | -1.6518 | 19.2% |
| ` lavender` | -0.5727 | 56.4% |
| ` and` | -0.0130 | 98.7% |
| ` deep` | -0.2860 | 75.1% |
| ` ind` | -0.1363 | 87.3% |
| `igo` | -0.0000 | 100.0% |
| `,` | -0.4006 | 67.0% |
| ` like` | -1.1882 | 30.5% |
| ` the` | -0.4078 | 66.5% |
| ` gentle` | -0.5544 | 57.4% |
| ` curve` | -0.3791 | 68.4% |
| ` of` | -0.0000 | 100.0% |
| ` a` | -0.0431 | 95.8% |
| ` par` | -0.6414 | 52.7% |
| `abolic` | -0.0016 | 99.8% |
| ` arc` | -0.0499 | 95.1% |
| ` just` | -1.2191 | 29.6% |
| ` before` | -0.0132 | 98.7% |
| ` it` | -0.0148 | 98.5% |
| ` meets` | -1.4061 | 24.5% |
| ` the` | -0.2248 | 79.9% |
| ` ground` | -0.5648 | 56.8% |
| `.` | -0.0416 | 95.9% |
| ` It` | -0.8257 | 43.8% |
| `’s` | -0.4762 | 62.1% |
| ` the` | -0.6356 | 53.0% |
| ` taste` | -1.2191 | 29.5% |
| ` of` | -0.0003 | 100.0% |
| ` a` | -0.2768 | 75.8% |
| ` cool` | -0.8105 | 44.5% |
| ` breeze` | -0.0455 | 95.6% |
| `,` | -1.2233 | 29.4% |
| ` whisper` | -1.3000 | 27.3% |
| `ing` | -0.0001 | 100.0% |
| ` secrets` | -0.6333 | 53.1% |
| ` of` | -0.1975 | 82.1% |
| ` a` | -1.7805 | 16.9% |
| ` storm` | -1.4896 | 22.5% |
| ` that` | -0.2647 | 76.7% |
| ` never` | -0.2621 | 76.9% |
| ` arrives` | -0.9235 | 39.7% |
| `,` | -0.0579 | 94.4% |
| ` a` | -1.4470 | 23.5% |
| ` delicate` | -1.6851 | 18.5% |
| ` balance` | -0.3177 | 72.8% |
| ` of` | -1.0825 | 33.9% |
| ` potential` | -1.4714 | 23.0% |
| ` energy` | -0.3717 | 69.0% |
| ` suspended` | -0.9083 | 40.3% |
| ` in` | -0.0097 | 99.0% |
| ` the` | -0.2390 | 78.7% |
| ` air` | -0.1538 | 85.7% |
| `.↵↵` | -0.2687 | 76.4% |
| `This` | -0.2081 | 81.2% |
| ` color` | -0.3918 | 67.6% |
| ` tastes` | -1.0448 | 35.2% |
| ` like` | -0.0147 | 98.5% |
| ` the` | -0.2369 | 78.9% |
| ` sweet` | -2.0821 | 12.5% |
| ` tang` | -0.3241 | 72.3% |
| ` of` | -0.0001 | 100.0% |
| ` dew` | -1.5099 | 22.1% |
| ` on` | -0.8759 | 41.6% |
| ` grass` | -0.6432 | 52.6% |
| `,` | -0.6397 | 52.7% |
| ` a` | -1.5995 | 20.2% |
| ` fleeting` | -0.9827 | 37.4% |
| ` probability` | -1.9240 | 14.6% |
| ` that` | -0.4123 | 66.2% |
| ` dances` | -0.7520 | 47.1% |
| ` between` | -0.8376 | 43.3% |
| ` certainty` | -0.5497 | 57.7% |
| ` and` | -0.0000 | 100.0% |
| ` chaos` | -0.3686 | 69.2% |
| `,` | -0.3301 | 71.9% |
| ` like` | -1.2672 | 28.2% |
| ` the` | -0.6711 | 51.1% |
| ` unpredictable` | -1.4196 | 24.2% |
| ` patterns` | -1.2395 | 29.0% |
| ` of` | -0.0229 | 97.7% |
| ` a` | -0.6435 | 52.5% |
| ` fract` | -0.0466 | 95.4% |
| `al` | -0.0000 | 100.0% |
| `.` | -1.0648 | 34.5% |
| ` It` | -0.3218 | 72.5% |
| `’s` | -0.3513 | 70.4% |
| ` the` | -0.9453 | 38.9% |
| ` ephemeral` | -1.7024 | 18.2% |
| ` flavor` | -2.0273 | 13.2% |
| ` of` | -0.0024 | 99.8% |
| ` a` | -0.4525 | 63.6% |
| ` gentle` | -1.9259 | 14.6% |
| ` rain` | -0.2121 | 80.9% |
| `,` | -0.3767 | 68.6% |
| ` each` | -1.0077 | 36.5% |
| ` drop` | -0.1437 | 86.6% |
| ` a` | -0.0973 | 90.7% |
| ` note` | -1.5102 | 22.1% |
| ` in` | -0.0149 | 98.5% |
| ` a` | -0.1779 | 83.7% |
| ` sym` | -0.2369 | 78.9% |
| `phony` | -0.0699 | 93.2% |
| ` of` | -0.1633 | 84.9% |
| ` fleeting` | -2.2685 | 10.3% |
| ` moments` | -0.4837 | 61.7% |
| `,` | -0.1057 | 90.0% |
| ` harmon` | -1.3813 | 25.1% |
| `izing` | -0.0019 | 99.8% |
| ` in` | -1.4513 | 23.4% |
| ` a` | -0.4932 | 61.1% |
| ` perfect` | -1.7490 | 17.4% |
| ` equation` | -0.9420 | 39.0% |
| ` of` | -0.4907 | 61.2% |
| ` warmth` | -1.3905 | 24.9% |
| ` and` | -0.0065 | 99.4% |
| ` cool` | -0.3096 | 73.4% |
| `ness` | -0.0083 | 99.2% |
| `.↵↵` | -0.5577 | 57.3% |
| `As` | -0.4150 | 66.0% |
| ` it` | -0.3987 | 67.1% |
| ` fades` | -1.2799 | 27.8% |
| `,` | -0.1649 | 84.8% |
| ` it` | -0.5757 | 56.2% |
| ` leaves` | -0.1152 | 89.1% |
| ` a` | -0.7162 | 48.9% |
| ` trace` | -1.3175 | 26.8% |
| ` of` | -0.9622 | 38.2% |
| ` nostalgia` | -0.8080 | 44.6% |
| `,` | -0.4794 | 61.9% |
| ` like` | -0.6020 | 54.8% |
| ` the` | -0.2172 | 80.5% |
| ` last` | -0.5304 | 58.8% |
| ` digit` | -1.7676 | 17.1% |
| ` of` | -0.2051 | 81.5% |
| ` pi` | -0.3600 | 69.8% |
| `—` | -0.1895 | 82.7% |
| `inf` | -1.4846 | 22.7% |
| `initely` | -0.0430 | 95.8% |
| ` elusive` | -0.9220 | 39.8% |
| `,` | -0.5326 | 58.7% |
| ` yet` | -0.5610 | 57.1% |
| ` profoundly` | -1.2286 | 29.3% |
| ` present` | -1.3533 | 25.8% |
| ` for` | -0.9547 | 38.5% |
| ` that` | -0.5322 | 58.7% |
| ` single` | -1.2369 | 29.0% |
| ` second` | -0.5454 | 58.0% |
| `,` | -0.3948 | 67.4% |
| ` a` | -0.9114 | 40.2% |
| ` reminder` | -0.6701 | 51.2% |
| ` of` | -0.3869 | 67.9% |
| ` the` | -0.7393 | 47.7% |
| ` beauty` | -0.1237 | 88.4% |
| ` in` | -0.7979 | 45.0% |
| ` trans` | -0.6819 | 50.6% |
| `ience` | -0.0003 | 100.0% |
| `,` | -0.8005 | 44.9% |
| ` where` | -0.9701 | 37.9% |
| ` every` | -0.9756 | 37.7% |
| ` taste` | -0.5194 | 59.5% |
| ` is` | -0.2374 | 78.9% |
| ` a` | -0.2548 | 77.5% |
| ` variable` | -1.4246 | 24.1% |
| ` in` | -0.1772 | 83.8% |
| ` the` | -0.0842 | 91.9% |
| ` equation` | -1.0179 | 36.1% |
| ` of` | -0.0002 | 100.0% |
| ` dusk` | -0.2015 | 81.7% |
| `.` | -0.0681 | 93.4% |

**Perplexity:** `1.87`

---